# Stitching consistency at 3x3 FOV neighbourhoods

Revisits the camera-rotation/BigStitcher-style correction work
(`acquisition/camera_rotation.py`, `misc/correct_camera_rotation.ipynb`). That
work concluded a single global affine transform cannot correct real per-FOV
positioning jitter -- fixing it needs a genuine BigStitcher-style joint
position solve (`fit_global_positions`) -- but that solve's own sparse anchor
sampling mostly produces small disconnected "star" correspondence-graph
components: one anchor + up to 4 leaves, each leaf constrained by exactly ONE
measurement, so a solved leaf position is numerically identical to that one
correspondence's own measurement. There is no redundant, self-checking
measurement per FOV yet.

This notebook looks at that redundancy question directly, at a small,
inspectable scale, instead of across a whole experiment:

1. **4-connected alignment** -- register a centre FOV's up/down/left/right
   neighbours to it via overlap-border phase correlation
   (`camera_rotation.register_neighbor_pair`, reusing the same primitive the
   correction pipeline itself uses).
2. **Corner concordance** -- at each of the centre FOV's 4 corners, the
   corner region is implied by TWO different registrations (its "up"
   neighbour's own alignment, and its "left" neighbour's own alignment, for
   the up-left corner). If the grid were perfectly rigid these would agree
   exactly; comparing them directly measures how much they don't.
3. **Diagonal loop closure** -- for each corner's diagonal FOV, register it
   via BOTH of the corner's already-aligned 4-connected neighbours
   independently (e.g. the up-left diagonal FOV is registered once through
   "up", once through "left") and compare the two resulting positions -- the
   classic BigStitcher loop-closure check: a consistent grid should give the
   same answer both ways.
4. Do 1-3 on **both the bead channel** (frame 0 -- HAL's own fiducial/
   focus-lock reference frame) **and the DAPI channel** (a configurable
   z-index), and compare the two channels' own registrations against each
   other -- do two independent signal sources agree on the same real
   misalignment?

**Multiple real datasets in one run** (`DATASETS` in Section 2): the same
pipeline above runs once per dataset, each cached under its own
`SAMPLE_DIR/analysis/`, so the 4 metrics are directly comparable across
datasets -- Section 13 uses that to test a specific hypothesis: a prior
investigation into `BC555_sample_05`'s `epi` vs. `disk` acquisitions found
`disk`'s bead/fiducial channel much weaker than `epi`'s (and DAPI relatively
stronger there) -- if that's the real mechanism, `disk`'s own corner/
diagonal residuals (computed identically for both channels above) should
show DAPI clearly beating beads, unlike whatever `epi` shows.

Repeated for **3 independent (non-overlapping) 3x3 neighbourhoods** per
dataset, each centred on a randomly-chosen FOV near the tissue centroid, so
no single neighbourhood's own idiosyncrasies (weak signal, a real local
defect) drives the whole conclusion.

**Why no SLURM submission machinery here** (unlike
`07_cluster_submit_analysis.ipynb`/`cli_analyze_fov.py`): this notebook reads
at most 3 neighbourhoods x 9 FOVs x 2 frames (bead + one DAPI z-plane) = 54
small, targeted reads per dataset -- a tiny fraction of that pipeline's own
per-FOV cost (a full multi-frame z-stack, budgeted at 2h/SLURM-task). The
intent is that this notebook's own Jupyter kernel already runs ON the
cluster (e.g. a login or interactive compute-node session) where the real
data lives, reading directly -- no separate job submission adds anything
here.

**Verification note**: this notebook was originally authored and verified
end-to-end against a small SYNTHETIC fixture (fake multi-FOV `.zarr` stacks
with known, injected shifts -- see `cache/scripts/build_test_stitching_fixture.py`,
gitignored), since no real cluster dataset was reachable from the authoring
environment. The synthetic run confirms the registration/corner/diagonal
logic executes correctly end-to-end and recovers injected shifts -- it does
NOT confirm anything about a real dataset's real camera-vs-stage alignment.
Review the results critically on real data before drawing conclusions from
them.

**Outputs**: per dataset, `SAMPLE_DIR/analysis/cache/test_stitching/*.csv`
(cardinal/corner/diagonal result tables) and
`SAMPLE_DIR/analysis/figures/test_stitching.*.png` (overview + per-
neighbourhood corner/diagonal overlay figures); plus a cross-dataset
`test_stitching.epi_vs_disk_beads_vs_dapi.png` comparison figure under this
repo's own gitignored cache folder (Section 13).

## 1 — Setup

In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.spatial import KDTree
from scipy.ndimage import shift as ndi_shift

# notebooks/tests/<subfolder>/ is three levels under the repo root (MERci/),
# same convention as notebooks/before_imaging/regular/ (3 levels).
MERCI_DIR = Path(os.getcwd()).parent.parent.parent   # MERci/

sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.io              import load_positions, read_image_frames
from MERci.common.experiment_info import load_experiment_info, resolve_sample_identity, positions_file_tag
from MERci.acquisition.configs    import (
    find_frame_table_for_hal_config, get_camera_pixel_size_um, get_camera_frame_size,
    get_all_color_frame_indices,
)
from MERci.acquisition.positions       import find_3x3_block
from MERci.acquisition.camera_rotation import (
    apply_microscope_orientation, crop_overlap, register_neighbor_pair,
)
from MERci.acquisition.alignment  import remove_hot_pixels, phase_drift
from MERci.acquisition.configs import load_microscope_orientation
from MERci.progress_display       import ProgressReporter
from MERci.visualization import get_merci_figures_dir

NOTEBOOK_NAME = "test_stitching"
print(f"MERCI_DIR : {MERCI_DIR}")

## 2 — Parameters

In [ ]:
# This notebook compares MULTIPLE real datasets in one run (originally
# analyzed just epi) -- one (label, path) entry per dataset to analyze, each
# run through the identical pipeline below and cached under its own
# SAMPLE_DIR/analysis/ (unrelated to wherever this MERci clone itself lives;
# it's the shared template repo, not inside either experiment folder).
DATASETS = [
    ("epi",  "/n/holylfs05/LABS/zhuang_lab/Lab/shared/projects/breast_cancer/experiments/BC555_sample_05/epi"),
    ("disk", "/n/holylfs05/LABS/zhuang_lab/Lab/shared/projects/breast_cancer/experiments/BC555_sample_05/disk"),
]

IMAGE_SUFFIX = ".zarr"   # must match what HAL wrote

# The bead (fiducial/focus-lock) frame is always the raw camera frame at
# index 0 of the cells-round stack -- HAL's own convention (see CLAUDE.md's
# "beads (frame 0)" / EXCLUDED_COLORS discussion in round_mosaics.ipynb).
BEAD_FRAME_INDEX = 0

# DAPI channel + z-INDEX (not a z position in um) -- the 4th z-step (0-indexed)
# of DAPI_COLOR_NM's own z-sweep in the cells-round frame table, mirroring
# "beads (frame 0)"'s own index convention. Adjust DAPI_COLOR_NM if some
# dataset's cells round doesn't use 405nm for DAPI.
DAPI_COLOR_NM = 405.0
DAPI_Z_INDEX  = 3

CHANNELS = ("beads", "dapi")

# How many independent (non-overlapping) 3x3 neighbourhoods to analyze, per dataset.
N_NEIGHBORHOODS = 3

# Candidate centre FOVs are drawn from this fraction of all cells-round FOVs
# closest to the tissue centroid (i.e. genuinely "near the centre of the
# tissue", not just anywhere with a complete 3x3 neighbourhood), then
# shuffled -- deterministic by default (SEED fixed) for reproducible re-runs;
# set SEED=None for a fresh random pick each run.
CENTRAL_CANDIDATE_FRACTION = 0.3
SEED = 0

TOLERANCE_FRACTION = 0.25   # same default as find_grid_neighbor/find_exterior_fovs
UPSAMPLE_FACTOR     = 10    # sub-pixel registration precision (1/UPSAMPLE_FACTOR px)

FORCE_RECOMPUTE = False   # set True to re-register even if a cache already exists

NEIGHBORHOOD_COLORS = ["tab:red", "tab:blue", "tab:green", "tab:orange", "tab:purple"]

# Explicit plot font sizes (NOTEBOOK_GUIDELINES.md #5) -- same values used
# throughout this repo's other notebooks.
PLOT_TITLE_FONTSIZE  = 14
PLOT_LABEL_FONTSIZE  = 12
PLOT_TICK_FONTSIZE   = 11
PLOT_LEGEND_FONTSIZE = 10

print(f"Datasets to analyze: {[label for label, _ in DATASETS]}")

## 3 — Resolve the cells round + FOV geometry

Same pattern as `misc/correct_camera_rotation.ipynb` section 3: build
`ExperimentMetadata` for the cells round, then measure `STEP_SIZE_UM`/
`OVERLAP_FRACTION` from the REAL positions file (median nearest-neighbour
distance) rather than trusting `ExperimentConfig`'s own
`pixel_size_um`/`image_size_px`/`non_overlap_fraction` formula, which that
notebook confirmed can silently disagree with the real camera/grid and break
every registration.

Wrapped in `resolve_dataset_geometry` so it runs once per entry in
`DATASETS` (Section 7) rather than for a single hardcoded dataset.

In [ ]:
def resolve_dataset_geometry(dataset_label, dataset_dir):
    '''Per-dataset setup: resolve sample identity, the cells round, camera
    geometry, and the REAL positions-derived step size/overlap fraction --
    everything downstream (frame loading, neighbourhood search, registration)
    needs. Logic is unchanged from this notebook's original single-dataset
    version -- just wrapped so it can run once per dataset in DATASETS.'''
    sample_dir = Path(dataset_dir)
    # resolve_sample_identity only looks at path component NAMES (no existence
    # check needed) -- appending a synthetic "MERci" gives the same split/flat
    # resolution logic it uses for the normal (auto-detected) case, even when
    # sample_dir isn't itself the folder this MERci clone lives in.
    sample_name, imaging_dir = resolve_sample_identity(sample_dir / "MERci")
    positions_tag = positions_file_tag(sample_name, imaging_dir)   # matches other notebooks' own naming

    info       = load_experiment_info(sample_dir / "metadata" / "experiment_info.yaml")
    microscope = info.microscope

    config = ExperimentConfig.from_sample_dir(
        sample_dir,
        positions_txt  = sample_dir / "positions" / f"positions_{positions_tag}.txt",
        image_suffix   = IMAGE_SUFFIX,
        microscope     = microscope,
    )
    meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                    image_suffix=config.image_suffix)

    cells_round_id = meta.round_for_imaging_type("cells")
    if not meta.round_fully_written(cells_round_id):
        print(f"WARNING [{dataset_label}]: cells round {cells_round_id} is not yet fully written on disk -- "
              f"some FOVs sampled below may be missing.")

    cells_series = next(s for s in meta.series_for_round(cells_round_id) if s.hal_config)
    hal_path     = Path(config.settings_dir) / cells_series.hal_config
    ft_path      = find_frame_table_for_hal_config(hal_path, config.metadata_dir)
    if ft_path is None or not ft_path.exists():
        raise FileNotFoundError(f"No frame table found for the cells round (hal_config={hal_path}).")
    frame_table = pd.read_csv(ft_path, index_col=0)

    dapi_frame_indices = get_all_color_frame_indices(frame_table, DAPI_COLOR_NM)
    if DAPI_Z_INDEX >= len(dapi_frame_indices):
        raise ValueError(
            f"DAPI_Z_INDEX={DAPI_Z_INDEX} out of range -- {DAPI_COLOR_NM:.0f}nm only has "
            f"{len(dapi_frame_indices)} z-plane(s) in this frame table.")
    dapi_frame_index = dapi_frame_indices[DAPI_Z_INDEX]

    print(f"[{dataset_label}] SAMPLE_DIR   : {sample_dir}")
    print(f"[{dataset_label}] SAMPLE_NAME  : {sample_name}  (IMAGING_DIR={imaging_dir!r})")
    print(f"[{dataset_label}] MICROSCOPE   : {microscope}")
    print(f"[{dataset_label}] Cells round  : {cells_round_id}  (series pattern: {cells_series.name!r})")
    print(f"[{dataset_label}] Frame table  : {ft_path}")
    print(f"[{dataset_label}] Bead frame   : index {BEAD_FRAME_INDEX}  "
          f"(color={frame_table.loc[BEAD_FRAME_INDEX, 'color']}, z={frame_table.loc[BEAD_FRAME_INDEX, 'z']}) "
          f"-- confirm this is really the bead/fiducial frame on real data.")
    print(f"[{dataset_label}] DAPI frame   : index {dapi_frame_index}  (z-index {DAPI_Z_INDEX} of "
          f"{len(dapi_frame_indices)} {DAPI_COLOR_NM:.0f}nm z-plane(s), z={frame_table.loc[dapi_frame_index, 'z']})")

    pixel_size_um      = get_camera_pixel_size_um(microscope)
    frame_width_px, _  = get_camera_frame_size(microscope)
    frame_width_um     = frame_width_px * pixel_size_um

    full_positions = load_positions(config.positions_txt)
    cells_fov_ids  = sorted(f for f in full_positions if f in meta.fovs)

    coords_arr = np.array([full_positions[f] for f in cells_fov_ids], dtype=float)
    centroid   = coords_arr.mean(axis=0)
    nn_dist, _ = KDTree(coords_arr).query(coords_arr, k=2)
    step_size_um     = float(np.median(nn_dist[:, 1]))
    overlap_fraction = max(0.0, 1.0 - step_size_um / frame_width_um)

    print(f"[{dataset_label}] Pixel size (um)    : {pixel_size_um}")
    print(f"[{dataset_label}] Frame width (um)   : {frame_width_um:.3f}  ({frame_width_px} px)")
    print(f"[{dataset_label}] Step size (um)     : {step_size_um:.3f}  (measured, median nearest-neighbour distance)")
    print(f"[{dataset_label}] Overlap fraction   : {overlap_fraction:.3f}")
    print(f"[{dataset_label}] FOVs in cells round: {len(cells_fov_ids)}")

    figures_dir = get_merci_figures_dir(sample_dir, "tests", NOTEBOOK_NAME, subfolder="fov_stitching")
    figures_dir.mkdir(parents=True, exist_ok=True)

    return {
        "dataset_label": dataset_label, "sample_dir": sample_dir, "sample_name": sample_name,
        "imaging_dir": imaging_dir, "microscope": microscope,
        "config": config, "meta": meta, "cells_round_id": cells_round_id, "cells_series": cells_series,
        "frame_table": frame_table, "dapi_frame_index": dapi_frame_index,
        "pixel_size_um": pixel_size_um, "frame_width_px": frame_width_px, "frame_width_um": frame_width_um,
        "full_positions": full_positions, "cells_fov_ids": cells_fov_ids,
        "coords_arr": coords_arr, "centroid": centroid,
        "step_size_um": step_size_um, "overlap_fraction": overlap_fraction,
        "figures_dir": figures_dir,
    }

## 4 — Microscope orientation

Reads this microscope's verified `transpose`/`flip_horizontal`/
`flip_vertical` convention straight from its MERlin microscope-parameters
JSON (`data/configs/merlin/microscope/*.json`), same as
`misc/correct_camera_rotation.ipynb` section 4 -- every raw frame is
reoriented via `apply_microscope_orientation` immediately after reading,
before any registration.

In [ ]:
def make_frame_loader(dataset_geo):
    '''Reads dataset_geo's own microscope orientation, then returns a
    load_channel_frames(fov_id) closure that returns
    {"beads": oriented bead frame, "dapi": oriented DAPI frame} for one FOV
    of THIS dataset -- one file open, both frames read together
    (read_image_frames is selective).

    NOTE: orientation is applied exactly ONCE, inside this closure -- every
    frame this notebook ever registers/crops is already oriented by the time
    it's used, so register_cardinal/register_chained below are called with
    NO orient_* kwargs (they default to False/no-op). Re-applying
    apply_microscope_orientation a SECOND time on an already-oriented image
    is not a no-op (transpose+flip is not idempotent) -- it silently
    corrupts every registration into measuring garbage.'''
    orientation = load_microscope_orientation(dataset_geo["microscope"],
                                               MERCI_DIR / "data" / "configs" / "merlin" / "microscope")
    transpose      = bool(orientation.get("transpose", False))
    flip_horizontal = bool(orientation.get("flip_horizontal", False))
    flip_vertical   = bool(orientation.get("flip_vertical", False))
    print(f"[{dataset_geo['dataset_label']}] transpose={transpose}  "
          f"flip_horizontal={flip_horizontal}  flip_vertical={flip_vertical}")

    cells_series, config = dataset_geo["cells_series"], dataset_geo["config"]
    dapi_frame_index = dataset_geo["dapi_frame_index"]

    def load_channel_frames(fov_id):
        path = cells_series.resolve_path(fov_id, config.image_suffix)
        bead_raw, dapi_raw = read_image_frames(
            path, [BEAD_FRAME_INDEX, dapi_frame_index],
            frame_width=config.frame_width, frame_height=config.frame_height,
        )
        return {
            "beads": apply_microscope_orientation(bead_raw, transpose=transpose, flip_horizontal=flip_horizontal, flip_vertical=flip_vertical),
            "dapi":  apply_microscope_orientation(dapi_raw, transpose=transpose, flip_horizontal=flip_horizontal, flip_vertical=flip_vertical),
        }

    return load_channel_frames

## 5 — Find 3 independent 3x3 (8-connected) neighbourhoods

`MERci.acquisition.positions.find_3x3_block` finds a centre FOV's full 3x3, 8-connected neighbourhood -- 4 cardinal
neighbours plus their shared diagonal FOVs (each diagonal resolved via
EITHER of its two cardinal neighbours, cross-checked against each other).
Candidate centres are the FOVs closest to the tissue centroid (a genuine
"near the centre of the tissue" pool), shuffled by `SEED` for a random pick;
each accepted neighbourhood's 9 FOVs are removed from the candidate pool
before picking the next one, so the 3 neighbourhoods never share a FOV.

In [ ]:
def find_neighborhoods(dataset_geo):
    '''3 independent (non-overlapping) 3x3 neighbourhoods for one dataset --
    candidate centres are the FOVs closest to the tissue centroid, shuffled
    by SEED for a random pick.'''
    dataset_label  = dataset_geo["dataset_label"]
    cells_fov_ids  = dataset_geo["cells_fov_ids"]
    coords_arr     = dataset_geo["coords_arr"]
    centroid       = dataset_geo["centroid"]
    full_positions = dataset_geo["full_positions"]
    step_size_um   = dataset_geo["step_size_um"]

    dist_to_centroid = np.hypot(coords_arr[:, 0] - centroid[0], coords_arr[:, 1] - centroid[1])
    order = np.argsort(dist_to_centroid)
    n_candidates = max(N_NEIGHBORHOODS * 9, int(round(len(cells_fov_ids) * CENTRAL_CANDIDATE_FRACTION)))
    central_candidates = [cells_fov_ids[i] for i in order[:n_candidates]]

    rng = np.random.default_rng(SEED)
    shuffled = list(central_candidates)
    rng.shuffle(shuffled)

    used_fovs = set()
    neighborhoods = []
    for candidate in shuffled:
        if candidate in used_fovs:
            continue
        block = find_3x3_block([candidate], full_positions, step_size_um, TOLERANCE_FRACTION)
        if block is None:
            continue
        block_fovs = set(block.values())
        if block_fovs & used_fovs:
            continue
        neighborhoods.append(block)
        used_fovs |= block_fovs
        if len(neighborhoods) == N_NEIGHBORHOODS:
            break

    if len(neighborhoods) < N_NEIGHBORHOODS:
        raise RuntimeError(
            f"[{dataset_label}] Only found {len(neighborhoods)}/{N_NEIGHBORHOODS} independent 3x3 "
            f"neighbourhoods near the tissue centroid -- try raising CENTRAL_CANDIDATE_FRACTION.")

    for i, block in enumerate(neighborhoods):
        print(f"[{dataset_label}] Neighbourhood {i}: center={block['center']}  {block}")

    return neighborhoods

## 6 — Registration helpers

- `register_cardinal`: registers a block's 4 cardinal neighbours directly to
  its centre (`register_neighbor_pair`, same primitive the correction
  pipeline uses).
- `corner_check`: at one corner (e.g. up-left), crops the two overlap bands
  that meet there (`crop_overlap` with the "up" and "left" directions), the
  intersection of which IS the corner region shared by all three FOVs. Each
  neighbour's own overlap-band crop is sub-pixel shifted onto the centre's
  frame using the pixel shift implied by its OWN direct registration above,
  then the two aligned corners are compared directly (`phase_drift`) -- if
  the grid is rigid these should coincide; the residual is exactly how much
  they don't.
- `register_chained`: registers *target_img* against *anchor_img* (whose own
  measured position may differ from its nominal grid position), returning
  the target's position CHAIN-composed through the anchor's own measured
  position (not just the raw pairwise registration).
- `diagonal_loop_closure`: for one corner's diagonal FOV, chains through
  BOTH of that corner's already-registered cardinal neighbours independently
  and compares the two resulting positions -- BigStitcher's loop-closure
  check.

`full_positions`/`overlap_fraction`/`pixel_size_um` are now explicit
parameters (not module globals) so the same functions work for whichever
dataset is currently in scope; `UPSAMPLE_FACTOR` stays a fixed global --
it's an algorithm setting, not a per-dataset one.

In [ ]:
CORNER_DIRS = {
    "up_left":    ("up", "left"),
    "up_right":   ("up", "right"),
    "down_left":  ("down", "left"),
    "down_right": ("down", "right"),
}
DIAGONAL_CHAIN = {
    "up_left":    (("up", "left"), ("left", "up")),
    "up_right":   (("up", "right"), ("right", "up")),
    "down_left":  (("down", "left"), ("left", "down")),
    "down_right": (("down", "right"), ("right", "down")),
}


def register_cardinal(block, frames, channel, full_positions, overlap_fraction, pixel_size_um):
    center_img = frames[block["center"]][channel]
    out = {}
    for direction in ("up", "down", "left", "right"):
        nb_fov = block[direction]
        nb_img = frames[nb_fov][channel]
        measured_xy, error = register_neighbor_pair(
            center_img, nb_img, full_positions[block["center"]], full_positions[nb_fov],
            direction, overlap_fraction, pixel_size_um, UPSAMPLE_FACTOR,
        )
        out[direction] = {
            "neighbor_fov": nb_fov, "nominal_xy": full_positions[nb_fov],
            "measured_xy": measured_xy, "error": error,
        }
    return out


def _pixel_shift_from_measured(measured_xy, nominal_xy, pixel_size_um):
    '''Back-solve the (dy_px, dx_px) phase-correlation shift register_neighbor_pair
    measured, from its own returned measured_xy -- measured_xy is always exactly
    nominal_xy + (dx_px, dy_px) * pixel_size_um (register_neighbor_pair's own algebra),
    so this recovers it without re-running phase_drift.'''
    dx_px = (measured_xy[0] - nominal_xy[0]) / pixel_size_um
    dy_px = (measured_xy[1] - nominal_xy[1]) / pixel_size_um
    return dy_px, dx_px


def corner_check(block, frames, channel, cardinal_results, corner_name, overlap_fraction, pixel_size_um):
    dir1, dir2 = CORNER_DIRS[corner_name]   # dir1 in {up,down}, dir2 in {left,right}
    center_img = frames[block["center"]][channel]
    img1, img2 = frames[block[dir1]][channel], frames[block[dir2]][channel]

    a1, n1 = crop_overlap(center_img, img1, dir1, overlap_fraction)   # a1,n1 shape (N, w)
    a2, n2 = crop_overlap(center_img, img2, dir2, overlap_fraction)   # a2,n2 shape (h, N)
    N = a1.shape[0]

    dy1, dx1 = _pixel_shift_from_measured(cardinal_results[dir1]["measured_xy"], cardinal_results[dir1]["nominal_xy"], pixel_size_um)
    dy2, dx2 = _pixel_shift_from_measured(cardinal_results[dir2]["measured_xy"], cardinal_results[dir2]["nominal_xy"], pixel_size_um)
    aligned1 = ndi_shift(n1.astype(float), shift=(dy1, dx1), order=1, mode="nearest")
    aligned2 = ndi_shift(n2.astype(float), shift=(dy2, dx2), order=1, mode="nearest")

    # a1's own row-range is crop_overlap's "up"/"down" convention: "up" takes the
    # anchor's LAST N rows, "down" takes its FIRST N rows (see crop_overlap's own
    # docstring) -- row_slice must select that SAME range back out of a2 (which
    # spans all h rows), so it's the OPPOSITE mapping from a naive dir1=="up"->first-N read.
    col_slice = slice(0, N) if dir2 == "left"  else slice(-N, None)
    row_slice = slice(-N, None) if dir1 == "up" else slice(0, N)

    center_corner   = a1[:, col_slice]
    corner_from_dir1 = aligned1[:, col_slice]
    corner_from_dir2 = aligned2[row_slice, :]

    # Sanity check -- both are literal slices of the SAME center_img, must match exactly.
    consistency_check = float(np.max(np.abs(center_corner.astype(float) - a2[row_slice, :].astype(float))))

    shift_px, err = phase_drift(remove_hot_pixels(corner_from_dir1), remove_hot_pixels(corner_from_dir2), UPSAMPLE_FACTOR)
    residual_um = float(np.hypot(*shift_px) * pixel_size_um)

    return {
        "residual_um": residual_um, "registration_error": err,
        "center_slice_consistency_maxdiff": consistency_check,
        "center_corner": center_corner, "corner_from_dir1": corner_from_dir1, "corner_from_dir2": corner_from_dir2,
        "dir1": dir1, "dir2": dir2,
    }


def register_chained(anchor_measured_xy, anchor_nominal_xy, anchor_img, target_img, target_nominal_xy, direction,
                      overlap_fraction, pixel_size_um):
    raw_measured_xy, error = register_neighbor_pair(
        anchor_img, target_img, anchor_nominal_xy, target_nominal_xy,
        direction, overlap_fraction, pixel_size_um, UPSAMPLE_FACTOR,
    )
    relative_offset = (raw_measured_xy[0] - anchor_nominal_xy[0], raw_measured_xy[1] - anchor_nominal_xy[1])
    chained_xy = (anchor_measured_xy[0] + relative_offset[0], anchor_measured_xy[1] + relative_offset[1])
    return chained_xy, error


def diagonal_loop_closure(block, frames, channel, cardinal_results, corner_name, full_positions,
                           overlap_fraction, pixel_size_um):
    (anchor1_key, dir1), (anchor2_key, dir2) = DIAGONAL_CHAIN[corner_name]
    diag_fov = block[corner_name]
    diag_img = frames[diag_fov][channel]

    anchor1_fov, anchor2_fov = block[anchor1_key], block[anchor2_key]
    via1_xy, err1 = register_chained(
        cardinal_results[anchor1_key]["measured_xy"], full_positions[anchor1_fov],
        frames[anchor1_fov][channel], diag_img, full_positions[diag_fov], dir1,
        overlap_fraction, pixel_size_um,
    )
    via2_xy, err2 = register_chained(
        cardinal_results[anchor2_key]["measured_xy"], full_positions[anchor2_fov],
        frames[anchor2_fov][channel], diag_img, full_positions[diag_fov], dir2,
        overlap_fraction, pixel_size_um,
    )
    residual_um = float(np.hypot(via1_xy[0] - via2_xy[0], via1_xy[1] - via2_xy[1]))
    return {
        "diag_fov": diag_fov, "anchor1_key": anchor1_key, "anchor2_key": anchor2_key,
        "via1_xy": via1_xy, "via2_xy": via2_xy, "error1": err1, "error2": err2,
        "residual_um": residual_um,
    }

## 7 — Run the pipeline (calculation, cached), per dataset

For each entry in `DATASETS`: loads all 9 FOVs' bead + DAPI frames per
neighbourhood, then runs 4-connected registration, corner concordance, and
diagonal loop closure on both channels. Cached to that dataset's own
`analysis/cache/test_stitching/*.csv` (`NOTEBOOK_GUIDELINES.md` #2/#3) --
re-running the notebook skips straight to display for a given dataset unless
`FORCE_RECOMPUTE=True`. Results for every dataset are collected into
`RESULTS` (keyed by dataset label), which every display section below loops
over.

In [ ]:
RESULTS = {}

for dataset_label, dataset_dir in DATASETS:
    print(f"\n=== {dataset_label} ({dataset_dir}) ===")
    dataset_geo = resolve_dataset_geometry(dataset_label, dataset_dir)
    load_channel_frames = make_frame_loader(dataset_geo)
    neighborhoods = find_neighborhoods(dataset_geo)

    config           = dataset_geo["config"]
    full_positions   = dataset_geo["full_positions"]
    overlap_fraction = dataset_geo["overlap_fraction"]
    pixel_size_um    = dataset_geo["pixel_size_um"]

    CACHE_DIR = config.analysis_dir / "cache" / NOTEBOOK_NAME
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cardinal_csv = CACHE_DIR / "cardinal_results.csv"
    corner_csv   = CACHE_DIR / "corner_results.csv"
    diagonal_csv = CACHE_DIR / "diagonal_results.csv"

    if not FORCE_RECOMPUTE and cardinal_csv.exists() and corner_csv.exists() and diagonal_csv.exists():
        cardinal_df = pd.read_csv(cardinal_csv)
        corner_df   = pd.read_csv(corner_csv)
        diagonal_df = pd.read_csv(diagonal_csv)
        corner_crops = {}   # not cached (image data) -- regenerate on demand for display below if needed
        all_frames = None   # ditto -- regenerated on demand (Section 10's corner-overlay display)
        print(f"[{dataset_label}] Loaded cached results from {CACHE_DIR}")
    else:
        cardinal_rows, corner_rows, diagonal_rows = [], [], []
        corner_crops = {}   # {(neighborhood_id, channel, corner_name): corner_check() result}

        reporter = ProgressReporter(total=len(neighborhoods) * 9, label=f"[{dataset_label}] Loading FOV frames")
        all_frames = []   # all_frames[i] = {fov_id: {"beads": img, "dapi": img}}
        for block in neighborhoods:
            frames = {}
            for fov_id in sorted(set(block.values())):
                frames[fov_id] = load_channel_frames(fov_id)
                reporter.update(1)
            all_frames.append(frames)
        reporter.done()

        for i, (block, frames) in enumerate(zip(neighborhoods, all_frames)):
            for channel in CHANNELS:
                cardinal_results = register_cardinal(block, frames, channel, full_positions, overlap_fraction, pixel_size_um)
                for direction, r in cardinal_results.items():
                    cardinal_rows.append({
                        "neighborhood": i, "channel": channel, "direction": direction,
                        "center_fov": block["center"], "neighbor_fov": r["neighbor_fov"],
                        "nominal_x": r["nominal_xy"][0], "nominal_y": r["nominal_xy"][1],
                        "measured_x": r["measured_xy"][0], "measured_y": r["measured_xy"][1],
                        "shift_um": float(np.hypot(r["measured_xy"][0] - r["nominal_xy"][0],
                                                    r["measured_xy"][1] - r["nominal_xy"][1])),
                        "error": r["error"],
                    })

                for corner_name in CORNER_DIRS:
                    cc = corner_check(block, frames, channel, cardinal_results, corner_name, overlap_fraction, pixel_size_um)
                    corner_crops[(i, channel, corner_name)] = cc
                    corner_rows.append({
                        "neighborhood": i, "channel": channel, "corner": corner_name,
                        "center_fov": block["center"],
                        "residual_um": cc["residual_um"], "registration_error": cc["registration_error"],
                        "center_slice_consistency_maxdiff": cc["center_slice_consistency_maxdiff"],
                    })

                for corner_name in DIAGONAL_CHAIN:
                    dc = diagonal_loop_closure(block, frames, channel, cardinal_results, corner_name,
                                                full_positions, overlap_fraction, pixel_size_um)
                    diagonal_rows.append({
                        "neighborhood": i, "channel": channel, "corner": corner_name,
                        "diag_fov": dc["diag_fov"], "anchor1_key": dc["anchor1_key"], "anchor2_key": dc["anchor2_key"],
                        "via1_x": dc["via1_xy"][0], "via1_y": dc["via1_xy"][1],
                        "via2_x": dc["via2_xy"][0], "via2_y": dc["via2_xy"][1],
                        "error1": dc["error1"], "error2": dc["error2"],
                        "residual_um": dc["residual_um"],
                    })

        cardinal_df = pd.DataFrame(cardinal_rows)
        corner_df   = pd.DataFrame(corner_rows)
        diagonal_df = pd.DataFrame(diagonal_rows)
        cardinal_df.to_csv(cardinal_csv, index=False)
        corner_df.to_csv(corner_csv, index=False)
        diagonal_df.to_csv(diagonal_csv, index=False)
        print(f"[{dataset_label}] Saved: {cardinal_csv}, {corner_csv}, {diagonal_csv}")

    # A cache hit doesn't carry a "dataset" column (older, single-dataset cache
    # schema) -- backfill it unconditionally so every downstream comparison
    # (Section 13) can rely on it regardless of which datasets hit cache.
    cardinal_df["dataset"] = dataset_label
    corner_df["dataset"]   = dataset_label
    diagonal_df["dataset"] = dataset_label

    print(f"[{dataset_label}] cardinal_df: {len(cardinal_df)} rows, corner_df: {len(corner_df)} rows, "
          f"diagonal_df: {len(diagonal_df)} rows")

    RESULTS[dataset_label] = {
        "dataset_geo": dataset_geo, "config": config,
        "full_positions": full_positions, "overlap_fraction": overlap_fraction, "pixel_size_um": pixel_size_um,
        "frame_width_um": dataset_geo["frame_width_um"], "sample_name": dataset_geo["sample_name"],
        "figures_dir": dataset_geo["figures_dir"], "centroid": dataset_geo["centroid"],
        "neighborhoods": neighborhoods, "load_channel_frames": load_channel_frames, "all_frames": all_frames,
        "cardinal_df": cardinal_df, "corner_df": corner_df, "diagonal_df": diagonal_df, "corner_crops": corner_crops,
    }

print(f"\nDatasets analyzed: {list(RESULTS.keys())}")

## 8 — Overview: the 3 neighbourhoods on the full FOV grid, per dataset

In [ ]:
for dataset_label, R in RESULTS.items():
    full_positions = R["full_positions"]
    cells_fov_ids  = R["dataset_geo"]["cells_fov_ids"]

    fig, ax = plt.subplots(figsize=(8, 8))
    all_xy = np.array([full_positions[f] for f in cells_fov_ids])
    ax.scatter(all_xy[:, 0], all_xy[:, 1], s=4, color="0.8", label="all cells-round FOVs", zorder=1)
    ax.scatter(*R["centroid"], marker="x", s=120, color="k", label="tissue centroid", zorder=4)

    for i, block in enumerate(R["neighborhoods"]):
        color = NEIGHBORHOOD_COLORS[i % len(NEIGHBORHOOD_COLORS)]
        xy = np.array([full_positions[f] for f in block.values()])
        ax.scatter(xy[:, 0], xy[:, 1], s=40, color=color, zorder=3,
                   label=f"neighbourhood {i} (center={block['center']})")
        cx, cy = full_positions[block["center"]]
        ax.scatter([cx], [cy], s=90, facecolors="none", edgecolors=color, linewidths=2, zorder=3)

    ax.invert_yaxis()
    ax.axis("equal")
    ax.set_xlabel("x (um)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_ylabel("y (um)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_title(f"{R['sample_name']}: 3 independent 3x3 neighbourhoods", fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    ax.legend(fontsize=PLOT_LEGEND_FONTSIZE, loc="best")
    fig.tight_layout()
    fig_path = R["figures_dir"] / f"{NOTEBOOK_NAME}.overview.png"
    fig.savefig(fig_path, dpi=150)
    plt.show()
    print(f"[{dataset_label}] Saved: {fig_path}")

## 9 — 4-connected registration results, beads vs. DAPI

Per-neighbourhood, per-direction shift magnitude (nominal vs. measured
position), plus a direct beads-vs-DAPI comparison: do the two independent
channels agree on the same real neighbour-to-neighbour shift?

In [ ]:
for dataset_label, R in RESULTS.items():
    cardinal_df = R["cardinal_df"]
    print(f"\n--- {dataset_label} ---")
    print(cardinal_df.pivot_table(index=["neighborhood", "direction"], columns="channel",
                                  values="shift_um").round(3))

    beads_wide = cardinal_df[cardinal_df.channel == "beads"].set_index(["neighborhood", "direction"])
    dapi_wide  = cardinal_df[cardinal_df.channel == "dapi"].set_index(["neighborhood", "direction"])
    channel_agreement = pd.DataFrame({
        "beads_measured_x": beads_wide["measured_x"], "beads_measured_y": beads_wide["measured_y"],
        "dapi_measured_x": dapi_wide["measured_x"], "dapi_measured_y": dapi_wide["measured_y"],
    })
    channel_agreement["beads_vs_dapi_um"] = np.hypot(
        channel_agreement["beads_measured_x"] - channel_agreement["dapi_measured_x"],
        channel_agreement["beads_measured_y"] - channel_agreement["dapi_measured_y"],
    )
    print(f"\n[{dataset_label}] Beads vs. DAPI agreement on the same neighbour-to-neighbour registration:")
    print(channel_agreement["beads_vs_dapi_um"].round(3).to_string())

In [ ]:
for dataset_label, R in RESULTS.items():
    cardinal_df = R["cardinal_df"]
    beads_wide = cardinal_df[cardinal_df.channel == "beads"].set_index(["neighborhood", "direction"])
    dapi_wide  = cardinal_df[cardinal_df.channel == "dapi"].set_index(["neighborhood", "direction"])
    channel_agreement = pd.DataFrame({
        "beads_measured_x": beads_wide["measured_x"], "beads_measured_y": beads_wide["measured_y"],
        "dapi_measured_x": dapi_wide["measured_x"], "dapi_measured_y": dapi_wide["measured_y"],
    })
    channel_agreement["beads_vs_dapi_um"] = np.hypot(
        channel_agreement["beads_measured_x"] - channel_agreement["dapi_measured_x"],
        channel_agreement["beads_measured_y"] - channel_agreement["dapi_measured_y"],
    )

    fig, ax = plt.subplots(figsize=(9, 5))
    labels = [f"nb{n}-{d}" for n, d in channel_agreement.index]
    ax.bar(labels, channel_agreement["beads_vs_dapi_um"], color="tab:purple")
    ax.set_ylabel("|beads - DAPI| measured position (um)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_title(f"{R['sample_name']}: beads vs. DAPI agreement per 4-connected registration",
                 fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(axis="x", labelsize=PLOT_TICK_FONTSIZE - 2, rotation=45)
    ax.tick_params(axis="y", labelsize=PLOT_TICK_FONTSIZE)
    fig.tight_layout()
    fig_path = R["figures_dir"] / f"{NOTEBOOK_NAME}.beads_vs_dapi.png"
    fig.savefig(fig_path, dpi=150)
    plt.show()
    print(f"[{dataset_label}] Saved: {fig_path}")

## 10 — Corner concordance

`residual_um` is how much the corner region disagrees depending on which of
the two adjacent 4-connected neighbours' alignment you trust -- 0 would mean
perfect grid rigidity. `center_slice_consistency_maxdiff` is a pure sanity
check (should be exactly 0.0 -- both `center_corner` extractions are literal
slices of the same image).

In [ ]:
for dataset_label, R in RESULTS.items():
    corner_df = R["corner_df"]
    print(f"\n--- {dataset_label} ---")
    print(corner_df.pivot_table(index=["neighborhood", "corner"], columns="channel",
                                values="residual_um").round(3))
    print(f"[{dataset_label}] Max center-slice consistency check (should be 0.0): "
          f"{corner_df['center_slice_consistency_maxdiff'].max()}")

In [ ]:
for dataset_label, R in RESULTS.items():
    corner_df = R["corner_df"]
    fig, ax = plt.subplots(figsize=(10, 5))
    pivot = corner_df.pivot_table(index=["neighborhood", "corner"], columns="channel", values="residual_um")
    x = np.arange(len(pivot))
    width = 0.35
    for offset, channel in zip((-width/2, width/2), CHANNELS):
        ax.bar(x + offset, pivot[channel], width=width, label=channel)
    ax.set_xticks(x)
    ax.set_xticklabels([f"nb{n}\n{c}" for n, c in pivot.index], fontsize=PLOT_TICK_FONTSIZE - 3)
    ax.set_ylabel("corner residual (um)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_title(f"{R['sample_name']}: corner concordance residual (0 = perfectly consistent)",
                 fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(axis="y", labelsize=PLOT_TICK_FONTSIZE)
    ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)
    fig.tight_layout()
    fig_path = R["figures_dir"] / f"{NOTEBOOK_NAME}.corner_residuals.png"
    fig.savefig(fig_path, dpi=150)
    plt.show()
    print(f"[{dataset_label}] Saved: {fig_path}")

### Corner overlay pictures

Per dataset, per neighbourhood, per channel: the corner region as implied by
each of the two adjacent neighbours' own alignment (red / green), plus the
centre FOV's own true corner pixels (blue) for reference -- well-aligned
corners blend toward white/gray; a real disagreement shows as separated
red/green ghosting. Regenerated on demand from that dataset's own
`neighborhoods`/`all_frames` (not cached -- image arrays, not scalars) if a
cache was loaded above.

In [ ]:
def _normalize01(img):
    lo, hi = np.percentile(img, [1, 99.5])
    return np.clip((img.astype(np.float32) - lo) / max(hi - lo, 1e-6), 0, 1)


def corner_rgb_overlay(cc):
    return np.stack([_normalize01(cc["corner_from_dir1"]), _normalize01(cc["corner_from_dir2"]),
                      _normalize01(cc["center_corner"])], axis=-1)


for dataset_label, R in RESULTS.items():
    neighborhoods  = R["neighborhoods"]
    corner_crops   = R["corner_crops"]
    full_positions = R["full_positions"]
    overlap_fraction, pixel_size_um = R["overlap_fraction"], R["pixel_size_um"]

    if not corner_crops:
        # Cache was loaded from disk above with no image data -- recompute just the crops for display.
        reporter = ProgressReporter(total=len(neighborhoods) * 9, label=f"[{dataset_label}] Reloading FOV frames for display")
        all_frames = []
        for block in neighborhoods:
            frames = {}
            for fov_id in sorted(set(block.values())):
                frames[fov_id] = R["load_channel_frames"](fov_id)
                reporter.update(1)
            all_frames.append(frames)
        reporter.done()
        for i, (block, frames) in enumerate(zip(neighborhoods, all_frames)):
            for channel in CHANNELS:
                cardinal_results = register_cardinal(block, frames, channel, full_positions, overlap_fraction, pixel_size_um)
                for corner_name in CORNER_DIRS:
                    corner_crops[(i, channel, corner_name)] = corner_check(
                        block, frames, channel, cardinal_results, corner_name, overlap_fraction, pixel_size_um)
        R["all_frames"] = all_frames   # persist -- Section 11's diagonal-overlay display reuses it

    for i in range(len(neighborhoods)):
        for channel in CHANNELS:
            fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
            for ax, corner_name in zip(axes, CORNER_DIRS):
                cc = corner_crops[(i, channel, corner_name)]
                ax.imshow(corner_rgb_overlay(cc))
                ax.set_title(f"{corner_name} (red={cc['dir1']}, green={cc['dir2']})\n"
                             f"residual={cc['residual_um']:.3f} um", fontsize=PLOT_TICK_FONTSIZE - 1)
                ax.axis("off")
            # dir1/dir2 vary PER CORNER (up_left uses up/left, up_right uses up/right, ...) -- each
            # panel's own title above states which cardinal directions red/green actually are for
            # that corner; this legend only needs to explain the fixed role each color plays.
            handles = [mpatches.Patch(color=c, label=n) for n, c in
                       (("vertical neighbour (up or down)", "red"),
                        ("horizontal neighbour (left or right)", "green"),
                        ("center (reference)", "blue"))]
            fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=PLOT_LEGEND_FONTSIZE)
            fig.suptitle(f"{R['sample_name']}: neighbourhood {i} (center={neighborhoods[i]['center']}) -- "
                         f"{channel} corner overlays", fontsize=PLOT_TITLE_FONTSIZE)
            fig.tight_layout(rect=[0, 0.08, 1, 0.93])
            fig_path = R["figures_dir"] / f"{NOTEBOOK_NAME}.corners_nb{i}_{channel}.png"
            fig.savefig(fig_path, dpi=150)
            plt.show()
            print(f"[{dataset_label}] Saved: {fig_path}")

## 11 — Diagonal loop closure

For each corner's diagonal FOV, `residual_um` is the disagreement between
its position as registered via the "up"-side chain vs. the "left"-side
chain (or the equivalent pair for the other 3 corners) -- BigStitcher's
classic loop-closure consistency check: a rigid, consistent grid gives the
same answer either way.

In [ ]:
for dataset_label, R in RESULTS.items():
    print(f"\n--- {dataset_label} ---")
    print(R["diagonal_df"].pivot_table(index=["neighborhood", "corner"], columns="channel",
                                       values="residual_um").round(3))

In [ ]:
for dataset_label, R in RESULTS.items():
    diagonal_df = R["diagonal_df"]
    fig, ax = plt.subplots(figsize=(10, 5))
    pivot = diagonal_df.pivot_table(index=["neighborhood", "corner"], columns="channel", values="residual_um")
    x = np.arange(len(pivot))
    width = 0.35
    for offset, channel in zip((-width/2, width/2), CHANNELS):
        ax.bar(x + offset, pivot[channel], width=width, label=channel)
    ax.set_xticks(x)
    ax.set_xticklabels([f"nb{n}\n{c}" for n, c in pivot.index], fontsize=PLOT_TICK_FONTSIZE - 3)
    ax.set_ylabel("loop-closure residual (um)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_title(f"{R['sample_name']}: diagonal loop-closure residual (0 = perfectly consistent)",
                 fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(axis="y", labelsize=PLOT_TICK_FONTSIZE)
    ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)
    fig.tight_layout()
    fig_path = R["figures_dir"] / f"{NOTEBOOK_NAME}.diagonal_residuals.png"
    fig.savefig(fig_path, dpi=150)
    plt.show()
    print(f"[{dataset_label}] Saved: {fig_path}")

### Diagonal overlay pictures

Places the two anchor FOVs at their own measured positions (blue / cyan),
and the diagonal FOV TWICE, tinted red at its "via anchor 1"-chained
position and green at its "via anchor 2"-chained position -- a consistent
grid places both diagonal copies on top of each other (yellow-ish where they
overlap); a real loop-closure disagreement shows as a visible red/green
double-image.

In [ ]:
def place_and_blend(frames_dict, positions_um, colors, pixel_size_um):
    h, w = next(iter(frames_dict.values())).shape
    xs = [positions_um[n][0] for n in frames_dict]
    ys = [positions_um[n][1] for n in frames_dict]
    x_min = min(xs) - w * pixel_size_um / 2
    y_min = min(ys) - h * pixel_size_um / 2

    # Derive canvas size from the SAME rounded per-tile row0/col0 used to place
    # each tile (rather than independently rounding (x_max-x_min)/pixel_size_um)
    # -- otherwise the two roundings can disagree by a pixel and the canvas ends
    # up one pixel too small for a tile placed at the far edge.
    placements = {}
    for name in frames_dict:
        x_um, y_um = positions_um[name]
        col0 = int(round((x_um - w * pixel_size_um / 2 - x_min) / pixel_size_um))
        row0 = int(round((y_um - h * pixel_size_um / 2 - y_min) / pixel_size_um))
        placements[name] = (row0, col0)
    canvas_h = max(row0 + h for row0, col0 in placements.values())
    canvas_w = max(col0 + w for row0, col0 in placements.values())

    canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.float32)
    for name, img in frames_dict.items():
        norm = _normalize01(img)
        row0, col0 = placements[name]
        canvas[row0:row0 + h, col0:col0 + w] += norm[..., None] * np.array(colors[name], dtype=np.float32)
    return np.clip(canvas, 0, 1), x_min, y_min


def crop_at_um(canvas, x_min, y_min, x_um, y_um, half_um, pixel_size_um):
    row = int(round((y_um - y_min) / pixel_size_um))
    col = int(round((x_um - x_min) / pixel_size_um))
    half_px = int(round(half_um / pixel_size_um))
    h, w = canvas.shape[:2]
    r0, r1 = max(row - half_px, 0), min(row + half_px, h)
    c0, c1 = max(col - half_px, 0), min(col + half_px, w)
    return canvas[r0:r1, c0:c1]


DIAGONAL_COLORS = {"anchor1": (0.2, 0.2, 1.0), "anchor2": (0.2, 0.8, 0.8),
                   "diag_via1": (1.0, 0.0, 0.0), "diag_via2": (0.0, 1.0, 0.0)}


for dataset_label, R in RESULTS.items():
    neighborhoods  = R["neighborhoods"]
    full_positions = R["full_positions"]
    overlap_fraction, pixel_size_um = R["overlap_fraction"], R["pixel_size_um"]
    frame_width_um = R["frame_width_um"]
    diagonal_df    = R["diagonal_df"]
    all_frames     = R["all_frames"]
    figures_dir    = R["figures_dir"]
    sample_name    = R["sample_name"]

    # Tight -- 1/4 of a FOV's own width on each side of the corner (so a total
    # crop width of half a FOV), showing just the overlaid corner itself, not the
    # surrounding anchor-tile context (an earlier, wider version at 0.6x showed
    # both anchors for spatial context, but that made the actual channel-to-
    # channel correspondence at the corner harder to judge at a glance).
    DIAGONAL_ZOOM_HALF_UM = frame_width_um * 0.25

    def draw_boundary_diagram(i, block, channel, cardinal_results):
        """Whole-neighbourhood schematic, rectangle outlines only (no pixel data):
        every FOV's NOMINAL grid-position boundary (dashed gray) vs. its MEASURED
        boundary (solid, colored by role -- same "vertical/horizontal neighbour"
        + via1/via2 convention as the corner overlays above), at true physical
        (um) scale -- so the shift's size can be judged directly against the FOV's
        own footprint, not just read off a residual_um number."""
        fig, ax = plt.subplots(figsize=(6, 6))
        half = frame_width_um / 2

        def add_square(xy, color, linestyle, linewidth):
            x0, y0 = xy[0] - half, xy[1] - half
            ax.add_patch(mpatches.Rectangle((x0, y0), frame_width_um, frame_width_um, fill=False,
                                             edgecolor=color, linestyle=linestyle, linewidth=linewidth))

        # anchor1 is always "up"/"down" (the vertical neighbour), anchor2 is
        # always "left"/"right" (horizontal) -- see DIAGONAL_CHAIN's own
        # definition above; consistent across all 4 corners, so each cardinal
        # FOV only needs to be drawn once here, not once per corner.
        role_colors = {"up": DIAGONAL_COLORS["anchor1"], "down": DIAGONAL_COLORS["anchor1"],
                       "left": DIAGONAL_COLORS["anchor2"], "right": DIAGONAL_COLORS["anchor2"]}

        add_square(full_positions[block["center"]], "black", "-", 2.0)   # nominal == measured by construction

        for direction in ("up", "down", "left", "right"):
            r = cardinal_results[direction]
            add_square(r["nominal_xy"], "0.6", "--", 1.0)
            add_square(r["measured_xy"], role_colors[direction], "-", 1.5)

        diag_channel_df = diagonal_df[(diagonal_df.neighborhood == i) & (diagonal_df.channel == channel)]
        for corner_name in DIAGONAL_CHAIN:
            add_square(full_positions[block[corner_name]], "0.6", "--", 1.0)
            row = diag_channel_df[diag_channel_df.corner == corner_name].iloc[0]
            add_square((row["via1_x"], row["via1_y"]), DIAGONAL_COLORS["diag_via1"], "-", 1.2)
            add_square((row["via2_x"], row["via2_y"]), DIAGONAL_COLORS["diag_via2"], "-", 1.2)

        all_xy = [full_positions[f] for f in block.values()]
        xs, ys = [p[0] for p in all_xy], [p[1] for p in all_xy]
        margin = frame_width_um * 0.6
        ax.set_xlim(min(xs) - margin, max(xs) + margin)
        ax.set_ylim(max(ys) + margin, min(ys) - margin)   # inverted y -- matches this notebook's other image-space plots
        ax.set_aspect("equal")
        ax.set_xlabel("x (um)", fontsize=PLOT_LABEL_FONTSIZE)
        ax.set_ylabel("y (um)", fontsize=PLOT_LABEL_FONTSIZE)
        ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
        legend_handles = [
            mpatches.Patch(facecolor="none", edgecolor="0.6", linestyle="--", label="nominal (grid) position"),
            mpatches.Patch(facecolor="none", edgecolor="black", label="center (reference)"),
            mpatches.Patch(facecolor="none", edgecolor=DIAGONAL_COLORS["anchor1"], label="vertical neighbour (measured)"),
            mpatches.Patch(facecolor="none", edgecolor=DIAGONAL_COLORS["anchor2"], label="horizontal neighbour (measured)"),
            mpatches.Patch(facecolor="none", edgecolor=DIAGONAL_COLORS["diag_via1"], label="diagonal, via vertical anchor"),
            mpatches.Patch(facecolor="none", edgecolor=DIAGONAL_COLORS["diag_via2"], label="diagonal, via horizontal anchor"),
        ]
        ax.legend(handles=legend_handles, fontsize=PLOT_LEGEND_FONTSIZE - 1, loc="upper center",
                  bbox_to_anchor=(0.5, -0.15), ncol=2)
        ax.set_title(f"{sample_name}: neighbourhood {i} (center={block['center']}) -- {channel}\n"
                     f"nominal vs. measured FOV boundaries", fontsize=PLOT_TITLE_FONTSIZE)
        fig.tight_layout()
        fig_path = figures_dir / f"{NOTEBOOK_NAME}.diagonals_nb{i}_{channel}_boundaries.png"
        fig.savefig(fig_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"[{dataset_label}] Saved: {fig_path}")

    for i, block in enumerate(neighborhoods):
        for channel in CHANNELS:
            frames = all_frames[i]
            cardinal_results = register_cardinal(block, frames, channel, full_positions, overlap_fraction, pixel_size_um)
            fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
            for ax, corner_name in zip(axes, DIAGONAL_CHAIN):
                dc_row = diagonal_df[(diagonal_df.neighborhood == i) & (diagonal_df.channel == channel)
                                      & (diagonal_df.corner == corner_name)].iloc[0]
                anchor1_key, anchor2_key = dc_row["anchor1_key"], dc_row["anchor2_key"]
                anchor1_fov, anchor2_fov = block[anchor1_key], block[anchor2_key]
                diag_fov = block[corner_name]
                place_frames = {
                    "anchor1": frames[anchor1_fov][channel], "anchor2": frames[anchor2_fov][channel],
                    "diag_via1": frames[diag_fov][channel], "diag_via2": frames[diag_fov][channel],
                }
                place_positions = {
                    "anchor1": cardinal_results[anchor1_key]["measured_xy"],
                    "anchor2": cardinal_results[anchor2_key]["measured_xy"],
                    "diag_via1": (dc_row["via1_x"], dc_row["via1_y"]),
                    "diag_via2": (dc_row["via2_x"], dc_row["via2_y"]),
                }
                canvas, x_min, y_min = place_and_blend(place_frames, place_positions, DIAGONAL_COLORS, pixel_size_um)
                # The shared corner between center/anchor1/anchor2/diagonal is
                # the midpoint of center's and the diagonal FOV's own NOMINAL
                # (grid) positions -- NOT the midpoint of via1_xy/via2_xy, which
                # is essentially the diagonal tile's own center (via1 and via2
                # differ only by the tiny measured residual). At the previous,
                # wider 0.6x crop this didn't matter -- the crop was big enough
                # to reach the real corner anyway -- but at the tighter 0.25x
                # crop it centered squarely inside the diagonal tile's own
                # interior instead, showing a uniform diag_via1/diag_via2 blend
                # with no anchor1/anchor2 content at all (confirmed directly:
                # exactly the "only a yellow image in the center, no border
                # overlay" symptom reported).
                zoom_x = (full_positions[block["center"]][0] + full_positions[block[corner_name]][0]) / 2
                zoom_y = (full_positions[block["center"]][1] + full_positions[block[corner_name]][1]) / 2
                zoom = crop_at_um(canvas, x_min, y_min, zoom_x, zoom_y, DIAGONAL_ZOOM_HALF_UM, pixel_size_um)
                ax.imshow(zoom)
                ax.set_title(f"{corner_name}\nresidual={dc_row['residual_um']:.3f} um", fontsize=PLOT_TICK_FONTSIZE)
                ax.axis("off")
            handles = [mpatches.Patch(color=c, label=n) for n, c in DIAGONAL_COLORS.items()]
            fig.legend(handles=handles, loc="lower center", ncol=4, fontsize=PLOT_LEGEND_FONTSIZE)
            fig.suptitle(f"{sample_name}: neighbourhood {i} (center={block['center']}) -- "
                         f"{channel} diagonal loop-closure overlays", fontsize=PLOT_TITLE_FONTSIZE)
            fig.tight_layout(rect=[0, 0.08, 1, 0.93])
            fig_path = figures_dir / f"{NOTEBOOK_NAME}.diagonals_nb{i}_{channel}.png"
            fig.savefig(fig_path, dpi=150)
            plt.show()
            print(f"[{dataset_label}] Saved: {fig_path}")

            # Boundary-shift diagram, right after this (neighbourhood, channel)'s
            # 4 corner-zoom panels above -- see draw_boundary_diagram's own
            # docstring.
            draw_boundary_diagram(i, block, channel, cardinal_results)

## 12 — Do the deviation vectors align across neighbourhoods?

If a single global affine transform (translation/rotation/scale) could
realign the whole grid, the 4-connected registration deviation
(`measured_xy - nominal_xy`) for a given direction should look similar
across independent neighbourhoods -- same magnitude, same direction --
rather than being neighbourhood-specific jitter. Plots each neighbourhood's
own 4 deviation vectors (one per cardinal direction), all drawn from a
shared origin (as if that neighbourhood's own center FOV were shifted to
`(0, 0)`), colored by neighbourhood -- overlapping arrows across
neighbourhoods argues FOR a single global correction; scattered/divergent
ones argue against it. Per dataset.

In [ ]:
for dataset_label, R in RESULTS.items():
    cardinal_df   = R["cardinal_df"]
    neighborhoods = R["neighborhoods"]
    figures_dir   = R["figures_dir"]
    sample_name   = R["sample_name"]

    # Precompute every deviation vector once, both for plotting and for the
    # spread summary below.
    cardinal_df["dx"] = cardinal_df["measured_x"] - cardinal_df["nominal_x"]
    cardinal_df["dy"] = cardinal_df["measured_y"] - cardinal_df["nominal_y"]

    fig, axes = plt.subplots(1, len(CHANNELS), figsize=(7 * len(CHANNELS), 6.5), squeeze=False)
    for ax, channel in zip(axes[0], CHANNELS):
        chan_df = cardinal_df[cardinal_df.channel == channel]
        for i in range(len(neighborhoods)):
            color = NEIGHBORHOOD_COLORS[i % len(NEIGHBORHOOD_COLORS)]
            sub = chan_df[chan_df.neighborhood == i]
            for _, row in sub.iterrows():
                dx, dy = row["dx"], row["dy"]
                ax.annotate("", xy=(dx, dy), xytext=(0, 0),
                            arrowprops=dict(arrowstyle="->", color=color, lw=1.8, alpha=0.85))
                if i == 0:   # label each direction once (neighbourhood 0), not once per neighbourhood
                    direction_label = row["direction"]
                    ax.text(dx, dy, "  " + direction_label, fontsize=PLOT_TICK_FONTSIZE - 2, color="0.2")

        # ax.annotate's arrows are NOT included in matplotlib's own autoscaling
        # (unlike ax.plot/scatter) -- confirmed directly: without this, the axes
        # silently stayed at their empty default [0, 1] range and every arrow
        # above was invisible, off-frame. Set limits explicitly from this
        # channel's own real dx/dy range instead, plus a margin.
        margin = max(chan_df["dx"].abs().max(), chan_df["dy"].abs().max()) * 1.3
        ax.set_xlim(-margin, margin)
        ax.set_ylim(margin, -margin)   # inverted y -- matches this notebook's other image-space plots
        ax.axhline(0, color="0.85", lw=0.8, zorder=0)
        ax.axvline(0, color="0.85", lw=0.8, zorder=0)
        ax.set_aspect("equal")
        ax.set_xlabel("dx = measured - nominal (um)", fontsize=PLOT_LABEL_FONTSIZE)
        ax.set_ylabel("dy = measured - nominal (um)", fontsize=PLOT_LABEL_FONTSIZE)
        ax.set_title(channel, fontsize=PLOT_TITLE_FONTSIZE)
        ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)

    legend_handles = []
    for i in range(len(neighborhoods)):
        color = NEIGHBORHOOD_COLORS[i % len(NEIGHBORHOOD_COLORS)]
        center_fov_id = neighborhoods[i]["center"]
        label = "neighbourhood " + str(i) + " (center=" + str(center_fov_id) + ")"
        legend_handles.append(mpatches.Patch(color=color, label=label))
    fig.legend(handles=legend_handles, loc="lower center", ncol=len(neighborhoods), fontsize=PLOT_LEGEND_FONTSIZE)
    fig.suptitle(f"{sample_name}: 4-connected deviation vectors (measured - nominal), by neighbourhood",
                 fontsize=PLOT_TITLE_FONTSIZE)
    fig.tight_layout(rect=[0, 0.12, 1, 0.9])
    fig_path = figures_dir / f"{NOTEBOOK_NAME}.deviation_vectors.png"
    fig.savefig(fig_path, dpi=150)
    plt.show()
    print(f"[{dataset_label}] Saved: {fig_path}")

    # Quantify the visual alignment: per (channel, direction), how much does the
    # deviation vector actually vary ACROSS the 3 independent neighbourhoods --
    # a small spread relative to the mean vector's own magnitude is what "a
    # single global affine transform would fix this" looks like numerically.
    spread = cardinal_df.groupby(["channel", "direction"]).agg(
        mean_dx=("dx", "mean"), mean_dy=("dy", "mean"),
        std_dx=("dx", "std"), std_dy=("dy", "std"),
    ).round(3)
    spread["mean_magnitude_um"] = np.hypot(spread["mean_dx"], spread["mean_dy"]).round(3)
    spread["std_magnitude_um"] = np.hypot(spread["std_dx"], spread["std_dy"]).round(3)
    print(f"\n[{dataset_label}] Deviation vector: mean vs. spread ACROSS the 3 neighbourhoods (per channel, direction):")
    print(spread.to_string())

## 13 — Epi vs. disk: does DAPI beat beads where beads are weak?

A prior investigation into `LeastSquaresGlobalAlignment` on these same two
`BC555_sample_05` acquisitions found `disk`'s bead/fiducial channel much
weaker/sparser than `epi`'s, while `disk`'s DAPI channel has higher SNR than
`epi`'s -- and concluded that's the likely reason MERlin's own global
alignment is badly broken for `disk` but not `epi`. If beads really are the
weak link for `disk`, this notebook's own per-channel corner/diagonal
residuals (computed identically for every dataset above, Sections 10/11)
should show DAPI residuals well below beads for `disk` specifically --
either matching or reversing whatever the gap looks like for `epi`.

In [ ]:
comparison_rows = []
for metric_name, df_key in (("corner", "corner_df"), ("diagonal", "diagonal_df")):
    for dataset_label, R in RESULTS.items():
        df = R[df_key]
        for channel, sub in df.groupby("channel"):
            comparison_rows.append({
                "metric": metric_name, "dataset": dataset_label, "channel": channel,
                "mean_residual_um": sub["residual_um"].mean(),
                "median_residual_um": sub["residual_um"].median(),
                "n": len(sub),
            })
comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.round(3).to_string(index=False))

pivot = comparison_df.pivot_table(index=["metric", "dataset"], columns="channel", values="mean_residual_um")
pivot["dapi_minus_beads_um"] = pivot["dapi"] - pivot["beads"]
print("\nDAPI - beads mean residual (negative = DAPI more accurate for that metric/dataset):")
print(pivot.round(3))

comparison_figures_dir = MERCI_DIR / "cache" / "2026_09_02_0912" / "test_stitching_comparison"
comparison_figures_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
dataset_labels = list(RESULTS.keys())
for ax, metric_name in zip(axes, ("corner", "diagonal")):
    sub = comparison_df[comparison_df.metric == metric_name]
    x = np.arange(len(dataset_labels))
    width = 0.35
    for offset, channel in zip((-width / 2, width / 2), CHANNELS):
        vals = [sub[(sub.dataset == d) & (sub.channel == channel)]["mean_residual_um"].iloc[0] for d in dataset_labels]
        ax.bar(x + offset, vals, width=width, label=channel)
    ax.set_xticks(x)
    ax.set_xticklabels(dataset_labels, fontsize=PLOT_TICK_FONTSIZE)
    ax.set_ylabel("mean residual (um)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_title(f"{metric_name} residual: beads vs. DAPI", fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(axis="y", labelsize=PLOT_TICK_FONTSIZE)
    ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)
fig.suptitle("epi vs. disk: does DAPI beat beads where beads are weak?", fontsize=PLOT_TITLE_FONTSIZE)
fig.tight_layout()
fig_path = comparison_figures_dir / f"{NOTEBOOK_NAME}.epi_vs_disk_beads_vs_dapi.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved: {fig_path}")